# Get the growth rates of the immunocompetent mice

This file calculates the exponential growth rate of the immunocompetent B6 mice and generates Supplementary Table 2.

## Prep

Load required packages

In [10]:
%matplotlib widget
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os
from estimator import Estimator
from src import pyxfunc

## Define data and user-input parameters

Define the data file, config file, save files, and group names

In [2]:
data_file = "data/growth_data_df.csv"
admix_result_path = "results/growth/"
config = "config_files/config_km.json"
b6_groups = ["Grp. A1 B6 (100% C1)", "Grp. A2 B6 (80% C1; 20% C11)", "Grp. A3 B6 (50% C1; 50% C11)", "Grp. A4 B6 (20% C1; 80% C11)", "Grp. A5 B6 (100% C11)"]
exclude = [456, 458, 461, 464, 471, 474, 478, 482, 483, 426, 428, 429, 430, 434, 437, 438, 442, 443, 451, 642, 662]

Create an estimator object from the config file

In [11]:
es = Estimator(config)

Load the growth data

In [12]:
growth_df = pd.read_csv(data_file)
growth_df = growth_df[~(growth_df["id"].isin(exclude))]
growth_df.loc[growth_df["group"]=="1C", "group"] = "Grp. A1 B6 (100% C1)"
growth_df.loc[growth_df["group"]=="2C", "group"] = "Grp. A2 B6 (80% C1; 20% C11)"
growth_df.loc[growth_df["group"]=="3C", "group"] = "Grp. A3 B6 (50% C1; 50% C11)"
growth_df.loc[growth_df["group"]=="4C", "group"] = "Grp. A4 B6 (20% C1; 80% C11)"
growth_df.loc[growth_df["group"]=="5C", "group"] = "Grp. A5 B6 (100% C11)"
growth_df = growth_df[growth_df["group"].isin(b6_groups)]
es.growth_df = growth_df
growth_df

,group,id,day,size
0,Grp. A1 B6 (100% C1),424,7,25.725600
1,Grp. A1 B6 (100% C1),424,10,36.392700
2,Grp. A1 B6 (100% C1),424,14,46.803512
3,Grp. A1 B6 (100% C1),424,17,98.260390
4,Grp. A1 B6 (100% C1),424,21,112.673203
...,...,...,...,...
1125,Grp. A5 B6 (100% C11),721,28,179.820056
1126,Grp. A5 B6 (100% C11),721,32,270.999669
1127,Grp. A5 B6 (100% C11),721,35,376.139244
1128,Grp. A5 B6 (100% C11),721,39,804.311946


Make a list of the growth rates to test

In [4]:
g_list = [round(0.01+0.001*i, 3) for i in range(round((.15-0.01)/0.001+1))]

## Estimate growth rates

Function to estimate the growth rate for each mouse

In [8]:
def get_grs():
    results = [[0]]*(len(growth_df["id"].unique())*len(g_list))
    idx = 0
    for mid in growth_df["id"].unique():
        for g_idx in range(len(g_list)):
            init = [growth_df[(growth_df["id"]==mid) & (growth_df["day"]==7)]["size"].iloc[0], 0, 0]
            end_time = max(growth_df[growth_df["id"] == mid]["day"])
            sol = pyxfunc.run(init[0], init[1], init[2], 7, end_time, np.inf, 
                                es.cutoff_0_raw, es.cutoff_0_percent, init[2],
                                g_list[g_idx], 0, 0, 0, 0, 0, 0, 0, 0)
            day_idxs = [(d-7)*100 for d in growth_df[growth_df["id"]==mid]["day"]]
            sizes = growth_df[growth_df["id"] == mid]["size"].tolist()
            err_win = pyxfunc.get_error(day_idxs, sizes, sol, 0, es.max_loser_subline_percent)
            results[idx] = [growth_df[(growth_df["id"]==mid)]["group"].tolist()[0], mid, g_list[g_idx], err_win[0], err_win[1]]
            idx += 1
    results = pd.DataFrame(results, columns=["group", "id", "g", "error", "winner"])
    results_best = results.groupby("id")[["group", "g", "error"]].apply(lambda x: x[x["error"]==x["error"].min()]).reset_index()
    results_best = results_best.drop("level_1", axis=1)
    return results_best

Get the growth rates

In [13]:
results_b6 = get_grs()
results_b6

,id,group,g,error
0,424,Grp. A1 B6 (100% C1),0.062,1424.881130
1,425,Grp. A1 B6 (100% C1),0.048,1094.336786
2,427,Grp. A1 B6 (100% C1),0.052,950.764380
3,431,Grp. A2 B6 (80% C1; 20% C11),0.051,5741.269361
4,432,Grp. A2 B6 (80% C1; 20% C11),0.035,14978.745755
5,433,Grp. A2 B6 (80% C1; 20% C11),0.039,1802.389459
6,435,Grp. A2 B6 (80% C1; 20% C11),0.060,771.601520
7,436,Grp. A3 B6 (50% C1; 50% C11),0.051,19229.911190
8,439,Grp. A3 B6 (50% C1; 50% C11),0.032,4017.765119
9,440,Grp. A3 B6 (50% C1; 50% C11),0.043,5557.613127


In [ ]:
if admix_result_path:
    results_b6.to_csv(admix_result_path + "b6_growth_rates.csv")